# CityFlow Traffic Simulation
### فقط از منوی بالا **Runtime ← Run all** بزنید
---

In [ ]:
# ── مرحله ۱: همگام Git + نصب افزونه (فقط در صورت نیاز) ─────────────
import os, subprocess, hashlib, glob

REPO = '/content/repo'
BRANCH = 'main'

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

def glob_cityflow_so():
    return glob.glob(f'{REPO}/cityflow.cpython-*.so')

def native_build_fingerprint(repo):
    h = hashlib.sha256()
    paths = [f'{repo}/CMakeLists.txt', f'{repo}/install.sh']
    paths += sorted(glob.glob(f'{repo}/src/**/*.cpp', recursive=True))
    paths += sorted(glob.glob(f'{repo}/src/**/CMakeLists.txt', recursive=True))
    for p in paths:
        try:
            with open(p, 'rb') as f:
                h.update(os.path.basename(p).encode())
                h.update(f.read())
        except OSError:
            continue
    return h.hexdigest()

# برای حذف کامل /content/repo و شروع از صفر این را True کنید
FORCE_FULL_REINSTALL = False

if FORCE_FULL_REINSTALL:
    sh(f'rm -rf "{REPO}"')

if not os.path.isdir(os.path.join(REPO, '.git')):
    print('⏳ کلون مخزن...')
    sh(f'git clone https://github.com/persiagfx/cityflow.git "{REPO}" --quiet')
else:
    print('⏳ همگام‌سازی با GitHub (reset به آخرین origin/%s)...' % BRANCH)
    sh(f'git -C "{REPO}" fetch origin')
    sh(f'git -C "{REPO}" checkout {BRANCH}')
    sh(f'git -C "{REPO}" reset --hard origin/{BRANCH}')

fp = native_build_fingerprint(REPO)
marker = os.path.join(REPO, '.colab_native_fingerprint')
so_ok = len(glob_cityflow_so()) > 0

need_build = True
if so_ok and os.path.isfile(marker):
    try:
        with open(marker) as mf:
            if mf.read().strip() == fp:
                need_build = False
    except OSError:
        pass

if need_build:
    print('⏳ کامپایل CityFlow (~۳–۴ دقیقه؛ اولین بار یا بعد از تغییر C++/CMake)...')
    sh(f'bash "{REPO}/install.sh"')
    with open(marker, 'w') as mf:
        mf.write(fp)
else:
    print('✓ افزونهٔ Python همین نسخه قبلاً build شده؛ frontend با Git به‌روز شد.')

print('✓ مرحلهٔ ۱ تمام')

In [ ]:
# ── مرحله ۲: اجرای شبیه‌سازی ──────────────────────────
import sys, os, shutil

sys.path.insert(0, '/content/repo')
os.chdir('/content/repo')

import cityflow

STEPS = 300  # تعداد گام‌های شبیه‌سازی

eng = cityflow.Engine('examples/config.json', thread_num=4)
for i in range(STEPS):
    eng.next_step()

shutil.copy('examples/replay.txt',          'frontend/replay.txt')
shutil.copy('examples/replay_roadnet.json', 'frontend/replay_roadnet.json')

print(f'✓ Simulation done — {STEPS} steps | vehicles: {eng.get_vehicle_count()}')

In [ ]:
# ── مرحله ۳: نمایش ────────────────────────────────────
import subprocess, threading, time, re, os

URL_RE = re.compile(r'https://[a-zA-Z0-9_.-]+\.trycloudflare\.com')

def strip_ansi(b):
    return re.sub(rb'\x1b\[[0-9;]*[a-zA-Z]', b'', b).decode('utf-8', errors='ignore')

# HTTP server (پورت اشغال بود → خطا در ترد؛ یک بار Runtime restart کنید)
def run_server():
    subprocess.run(
        ['python3', '-m', 'http.server', '8765', '--directory', '/content/repo/frontend'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

threading.Thread(target=run_server, daemon=True).start()
time.sleep(1.5)

if not os.path.exists('/content/cloudflared'):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/content/cloudflared',
    ], check=True)
    subprocess.run(['chmod', '+x', '/content/cloudflared'])

out = {'url': None}

def drain_cloudflared(pipe):
    buf = b''
    try:
        while out['url'] is None:
            chunk = pipe.read(512)
            if not chunk:
                break
            buf = (buf + chunk)[-16384:]
            text = strip_ansi(buf)
            m = URL_RE.search(text)
            if m:
                out['url'] = m.group(0).rstrip('.,;')
                break
    finally:
        pipe.close()

proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8765'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
threading.Thread(target=drain_cloudflared, args=(proc.stdout,), daemon=True).start()

print('⏳ Creating link (تا ۹۰ ثانیه؛ اولین بار گاهی طول می‌کشد)...')
for _ in range(90):
    if out['url']:
        break
    time.sleep(1)

url = out['url']

print()
print('=' * 55)
print(f'  🌐 {url}')
print('=' * 55)
print()
if url is None:
    print('  ⚠ لینک پیدا نشد — یک بار دیگر همین سلول را اجرا کنید یا Runtime → Restart runtime.')
    print('  ⚠ اگر پورت ۸۷۶۵ اشغال است، Restart runtime لازم است.')
else:
    print('  ① لینک را در Chrome یا Firefox باز کنید')
    print('  ② Roadnet File  ←  replay_roadnet.json')
    print('  ③ Replay File   ←  replay.txt')
    print('  ④ دکمه Start را بزنید')
print()
print('  💡 تا زمانی که این سلول در حال اجراست، لینک فعال است')